In [1]:
using Revise
using GeometryBasics: Vec3f, Point3f, Cylinder
using LinearAlgebra
using GLMakie
using StaticArrays
using VMRobotControl
using VMRobotControl.Splines: CubicSpline
using DifferentialEquations
using MeshIO
include("../functions.jl")

circle_center_tangent_to_lines (generic function with 1 method)

# Building a simple 2 links arm robot

## Building the robot

### Rigid body tree

In [13]:
robot = Mechanism{Float64}("SimpleArmRobot")
F0 = root_frame(robot)
F1 = add_frame!(robot; id="L1_frame")
F2 = add_frame!(robot; id="L2_frame")
F_EE = add_frame!(robot; id="EE_frame")

X = SVector(1., 0., 0.)
Y = SVector(0., 1., 0.)
Z = SVector(0., 0., 1.)

J1 = Revolute(X)
J2 = Revolute(X, Transform(0.4*Z))
J_EE = Rigid(Transform(0.4*Z))

add_joint!(robot, J1; parent=F0, child=F1, id="J1")
add_joint!(robot, J2; parent=F1, child=F2, id="J2")
add_joint!(robot, J_EE; parent=F2, child=F_EE, id="J_EE")

"J_EE"

### Plotting 

In [14]:
r = compile(robot)
kcache = Observable(new_kinematics_cache(r))

fig = Figure(size=(800, 600))
ls = LScene(fig[1, 1]; show_axis=true)  # 3D interactive scene
cam3d!(ls)  
robotsketch!(ls, kcache; scale = 0.75)

display(fig)

GLMakie.Screen(...)

In [15]:
t = 0.0
q = Float64[0.5, 0.2]
kinematics!(kcache[], t, q)
notify(kcache)

false

### Adding mass and inertia

In [16]:
add_coordinate!(robot, FrameOrigin(F1); id="base_centre_of_mass")
add_coordinate!(robot, FrameOrigin(F2); id="elbow_centre_of_mass")
add_coordinate!(robot, FrameOrigin(F_EE); id="ee_centre_of_mass")

add_component!(robot, PointMass(1.0, "base_centre_of_mass"); id="base_mass")
add_component!(robot, PointMass(0.2, "elbow_centre_of_mass"); id="lower_arm_mass")
add_component!(robot, PointMass(0.1, "ee_centre_of_mass"); id="ee_mass")

I_mat = @SMatrix [
    0.1  0.    0.  ;
    0.    0.1  0.  ;
    0.    0.    0.1
]
add_inertia!(robot, F2, I_mat; id="L2_inertia")

2DOF Mechanism{Float64} "SimpleArmRobot" with 4 frames, 3 joints, 4 coordinates, 4 components

### Simulation

In [18]:
tspan = (0., 10.)
q_dot  = zero_q̇(r)

r = compile(robot)
dcache = new_dynamics_cache(r)
g = VMRobotControl.DEFAULT_GRAVITY
prob = get_ode_problem(dcache, g, q, q_dot, tspan)
sol = solve(prob, Tsit5(), abstol=1e-6, reltol=1e-6);

### Animation

In [20]:
fig = Figure(size=(700, 750))
ls = LScene(fig[1, 1]; show_axis = false)  # 3D interactive scene
cam = cam3d!(ls, camera=:perspective, center=false)  
cam.lookat[] = [-0.0, 0., 0.2]
cam.eyeposition[] = [-1.25, 0.88, 0.75]

plotting_kcache = Observable(new_kinematics_cache(compile(robot)))
robotsketch!(ls, plotting_kcache; scale = 0.5)

display(fig)
animate_robot_odesolution(fig, sol, plotting_kcache, "test.mp4"; fps = 25);

### Adding damping

In [21]:
for i = 1:2
    add_coordinate!(robot, JointSubspace("J$i"); id="J$i")
    add_component!(robot, LinearDamper(0.5, "J$i"); id="J$(i)_damper")
end

In [22]:
# SIMULATION
tspan = (0., 10.)
q_dot  = zero_q̇(r)
r = compile(robot)
dcache = new_dynamics_cache(r)
g = VMRobotControl.DEFAULT_GRAVITY
prob = get_ode_problem(dcache, g, q, q_dot, tspan)
sol = solve(prob, Tsit5(), abstol=1e-10, reltol=1e-10);

# ANIMATION
fig = Figure(size=(700, 750))
ls = LScene(fig[1, 1]; show_axis = false)  # 3D interactive scene
cam = cam3d!(ls, camera=:perspective, center=false)  
cam.lookat[] = [-0.0, 0., -0.05]
cam.eyeposition[] = [-1.25, 0.88, 0.6]

plotting_kcache = Observable(new_kinematics_cache(compile(robot)))
robotsketch!(ls, plotting_kcache; scale = 0.5)

display(fig)
animate_robot_odesolution(fig, sol, plotting_kcache, "test.mp4"; fps = 25);

## Controlling it with virtual mechanisms

### Adding virtual spring (and damper)

In [23]:
vms = VirtualMechanismSystem("myVMS", robot)

add_coordinate!(vms, FramePoint(".robot.root_frame", SVector(0.0, 1.0, 1.0));   id="Target position")
add_coordinate!(vms, CoordDifference(".robot.ee_centre_of_mass", "Target position"); id="Position error");

K = SMatrix{3, 3}(10., 0., 0., 0., 10., 0., 0., 0., 10.)
add_component!(vms, LinearSpring(K, "Position error"); id="Linear Spring");
D = SMatrix{3, 3}(5., 0., 0., 0., 5.0, 0., 0., 0., 5.)
add_component!(vms, LinearDamper(D, "Position error"); id="Linear Damper")

"Linear Damper"

In [24]:
# SIMULATION
tspan = (0., 10.)
vms_compiled = compile(vms)
q = (zero_q(vms_compiled.robot), zero_q(vms_compiled.virtual_mechanism)) 
q̇ = (zero_q̇(vms_compiled.robot), zero_q̇(vms_compiled.virtual_mechanism)) 
g = VMRobotControl.DEFAULT_GRAVITY
dcache = new_dynamics_cache(vms_compiled)
prob = get_ode_problem(dcache, g, q, q̇, tspan)
sol = solve(prob, Tsit5(), progress=true; maxiters=1e6, abstol=1e-6, reltol=1e-6);

# ANIMATION
fig = Figure(size=(700, 750))
ls = LScene(fig[1, 1]; show_axis = false)  # 3D interactive scene
cam = cam3d!(ls, camera=:perspective, center=false)  
cam.lookat[] = [0.2, 0.2, 0.2]
cam.eyeposition[] = [-1.25, 0.88, 0.7]

plotting_kcache = Observable(new_kinematics_cache(compile(robot)))
robotsketch!(ls, plotting_kcache; scale = 0.5)

display(fig)
animate_robot_odesolution(fig, sol, plotting_kcache, "test.mp4"; fps = 25);

### Simulating with perturbing force

In [25]:
disturbance_func(t) = mod(t, 6) < 3 ? SVector(0., 0., 0.) : SVector(0., 0., 3.)

f_setup(cache) = get_compiled_coordID(cache, ".robot.ee_centre_of_mass")

function f_control(cache, t, args, extra)
    tcp_pos_coord_id = args
    F = disturbance_func(t)
    uᵣ, uᵥ = get_u(cache)
    z = configuration(cache, tcp_pos_coord_id)
    J = jacobian(cache, tcp_pos_coord_id)
    mul!(uᵣ, J', F)
    nothing
end

# SIMULATION
tspan = (0., 12.)
vms_compiled = compile(vms)
q = (zero_q(vms_compiled.robot), zero_q(vms_compiled.virtual_mechanism)) 
q̇ = (zero_q̇(vms_compiled.robot), zero_q̇(vms_compiled.virtual_mechanism)) 
g = VMRobotControl.DEFAULT_GRAVITY
dcache = new_dynamics_cache(vms_compiled)
prob = get_ode_problem(dcache, g, q, q̇, tspan; f_setup, f_control)
sol = solve(prob, Tsit5(), progress=true; maxiters=1e6, abstol=1e-6, reltol=1e-6);

# ANIMATION
fig = Figure(size=(700, 750))
ls = LScene(fig[1, 1]; show_axis = false)  # 3D interactive scene
cam = cam3d!(ls, camera=:perspective, center=false)  
cam.lookat[] = [0.2, 0.2, 0.2]
cam.eyeposition[] = [-1.25, 0.88, 0.7]

plotting_t = Observable(0.0)
plotting_kcache = Observable(new_kinematics_cache(compile(robot)))
robotsketch!(ls, plotting_kcache; scale = 0.5)

tcp_pos_id = get_compiled_coordID(plotting_kcache[], "ee_centre_of_mass")
tcp_pos = map(plotting_kcache) do kcache
    Point3f(configuration(kcache, tcp_pos_id))
end
force = map(t -> 0.01 * Vec3f(disturbance_func(t)), plotting_t)
arrowsize = map(f -> 0.1*(f'*f)^(0.25), force)
arrows!(ls, map(p -> [p], tcp_pos), map(f -> [f], force); color = :red, arrowsize)

display(fig)
animate_robot_odesolution(fig, sol, plotting_kcache, "test.mp4"; fps = 25, t=plotting_t);

# ShadowHand Application

### URDF (Universal Robot Description Format) Loading

In [26]:
using FileIO, UUIDs
try
    FileIO.add_format(format"DAE", (), ".dae", [:DigitalAssetExchangeFormatIO => UUID("43182933-f65b-495a-9e05-4d939cea427d")])
catch
end

cfg = URDFParserConfig(;suppress_warnings=true) # This is just to hide warnings about unsupported URDF features
module_path = joinpath(splitpath(splitdir(pathof(VMRobotControl))[1])[1:end-1])
robot = parseURDF(joinpath(module_path, "URDFs/sr_description/sr_hand_vm_compatible.urdf"), cfg)

add_coordinate!(robot, FrameOrigin("rh_ffdistal"); id="rh_ffdistal")
add_coordinate!(robot, FrameOrigin("rh_mfdistal"); id="rh_mfdistal")
add_coordinate!(robot, FrameOrigin("rh_rfdistal"); id="rh_rfdistal")
add_coordinate!(robot, FrameOrigin("rh_lfdistal"); id="rh_lfdistal")
add_coordinate!(robot, FrameOrigin("rh_thdistal"); id="rh_thdistal")
add_coordinate!(robot, FrameOrigin("rh_ffproximal"); id="rh_ffproximal")
add_coordinate!(robot, FrameOrigin("rh_thmiddle"); id="rh_thmiddle")

m = compile(robot)
kcache = Observable(new_kinematics_cache(m))  
fig = Figure(size=(700, 600))
ls = LScene(fig[1, 1]; show_axis=true)
cam3d!(ls)  
robotvisualize!(ls, kcache)
display(fig)

add_gravity_compensation!(robot, VMRobotControl.DEFAULT_GRAVITY)
joint_limits = cfg.joint_limits

for joint_id in keys(joints(robot))
    limits = joint_limits[joint_id]
    isnothing(limits) && continue
    add_coordinate!(robot, JointSubspace(joint_id);  id="$(joint_id)_coord")
    @assert ~isnothing(limits.lower) && ~isnothing(limits.upper)
    add_deadzone_springs!(robot, 50.0, (limits.lower+0.1, limits.upper-0.1), "$(joint_id)_coord")
    add_component!(robot, LinearDamper(0.01, "$(joint_id)_coord"); id="$(joint_id)_damper")
end

### Simple Simulation

In [27]:
# SPRINGS AND DAMPERS
vms = VirtualMechanismSystem("myShadowVMS", robot)
root = root_frame(vms.robot)
add_coordinate!(vms, FramePoint(".robot.$root", SVector(0.0, -0.15, 0.35));        id="Target position")
add_coordinate!(vms, CoordDifference(".robot.rh_fftip_mass_coord", "Target position"); id="Position error");

K = SMatrix{3, 3}(10., 0., 0., 0., 10., 0., 0., 0., 10.)
add_component!(vms, LinearSpring(K, "Position error");  id="Linear Spring")
D = SMatrix{3, 3}(5., 0., 0., 0., 5.0, 0., 0., 0., 5.)
add_component!(vms, LinearDamper(D, "Position error");  id="Linear Damper")

add_component!(vms, LinearSpring(0.1, ".robot.rh_LFJ5_coord"); id = "lf j5 angular spring")

# SIMULATION 
tspan = (0., 6.)
vms_compiled = compile(vms)
q = (zero_q(vms_compiled.robot), Float64[]) # Robot joint angle, vm joint angles
q̇ = (zero_q̇(vms_compiled.robot), Float64[]) # Robot joint velocity, vm joint velocities
g = VMRobotControl.DEFAULT_GRAVITY
dcache = new_dynamics_cache(vms_compiled)
prob = get_ode_problem(dcache, g, q, q̇, tspan)
sol = solve(prob, Rosenbrock23(autodiff=false), progress=true; maxiters=1e6, abstol=1e-3, reltol=1e-3);

In [28]:
# ANIMATION
fig = Figure(size = (720, 720), figure_padding=0)
display(fig)
ls = LScene(fig[1, 1]; show_axis=false)
cam = cam3d!(ls, camera=:perspective, center=false)
cam.lookat[] = [-0.6, 0.0, 0.0]
cam.eyeposition[] = [0.87, 0.04, 0.41]

plotting_t = Observable(0.0)
plotting_kcache = Observable(new_kinematics_cache(compile(robot)))
robotvisualize!(ls, plotting_kcache;)

animate_robot_odesolution(fig, sol, plotting_kcache, "test.mp4"; t=plotting_t);

LoadError: Screen not open!

### Real Application

In [79]:
vms = VirtualMechanismSystem("myShadowVMS", robot)
root = root_frame(vms.robot)

cylinder_radius = 0.03 

cylinder_position = SVector(0.0, -0.05, 0.35)

add_coordinate!(vms,  ConstCoord(cylinder_position);  id="cylinder position")
add_coordinate!(vms, ConstCoord(cylinder_radius); id="cylinder radius")

add_coordinate!(vms, FramePoint(".robot.rh_palm", SVector(0. , 0., 0.07)); id="second palm point")

repulsed_frames = (".robot.rh_fftip_mass_coord", ".robot.rh_mftip_mass_coord", ".robot.rh_rftip_mass_coord",".robot.rh_lftip_mass_coord" , 
                    ".robot.rh_thtip_mass_coord", ".robot.rh_ffmiddle_mass_coord",".robot.rh_mfmiddle_mass_coord", ".robot.rh_rfmiddle_mass_coord",
                    ".robot.rh_lfmiddle_mass_coord",  ".robot.rh_thmiddle_mass_coord", ".robot.rh_ffproximal_mass_coord", ".robot.rh_mfproximal_mass_coord",
                    ".robot.rh_rfproximal_mass_coord", ".robot.rh_lfproximal_mass_coord", ".robot.rh_thproximal_mass_coord", ".robot.rh_palm_mass_coord", "second palm point",
                    ".robot.rh_ffdistal", ".robot.rh_mfdistal", ".robot.rh_rfdistal", ".robot.rh_lfdistal", ".robot.rh_thdistal", ".robot.rh_thmiddle")
repulsed_frames_names = ("fftip", "mftip", "rftip", "lftip", "thtip", "ffmiddle", "mfmiddle", "rfmiddle", "lfmiddle", "thmiddle", "ffprox", 
                "mfprox", "rfprox", "lfprox", "thprox", "palm", "palm2", "ffdistal", "mfdistal", "rfdistal", "lfdistal", "thdistal", "thmiddle2")

for i in 1:length(repulsed_frames)
    frame = repulsed_frames[i]
    add_coordinate!(vms, CoordDifference(frame, "cylinder position") ; id = "$(repulsed_frames_names[i]) cylinder diff" )
    add_coordinate!(vms, CoordSlice("$(repulsed_frames_names[i]) cylinder diff", SVector(2,3)); id="$(repulsed_frames_names[i]) planar error")
    add_coordinate!(vms, CoordNorm("$(repulsed_frames_names[i]) planar error") ; id = "$(repulsed_frames_names[i]) planar error norm")
    add_coordinate!(vms, CoordDifference("$(repulsed_frames_names[i]) planar error norm", "cylinder radius"); id = "shifted $(repulsed_frames_names[i]) cylinder error" )

    add_component!(vms, ReLUSpring(50.0, "shifted $(repulsed_frames_names[i]) cylinder error", true); id="$(repulsed_frames_names[i]) cylinder repulsive spring")
    add_component!(vms, RectifiedDamper(5.0, "$(repulsed_frames_names[i]) planar error norm", (0.0, 1.1*cylinder_radius), true, false); id="$(repulsed_frames_names[i]) cylinder damper")
end

In [80]:
add_coordinate!(vms, FramePoint(".robot.$root", SVector(0.03, -0.05, 0.34));        id="Target position")
add_coordinate!(vms, CoordDifference(".robot.rh_fftip_mass_coord", "Target position"); id="Position error");

K = SMatrix{3, 3}(0.5, 0., 0., 0., 0.5, 0., 0., 0., 0.5)
add_component!(vms, LinearSpring(K, "Position error");  id="Linear Spring")
D = SMatrix{3, 3}(0.1, 0., 0., 0., 0.1, 0., 0., 0., 0.1)
add_component!(vms, LinearDamper(D, "Position error");  id="Linear Damper")

add_component!(vms, LinearSpring(0.1, ".robot.rh_LFJ5_coord"); id = "lf j5 angular spring")

# SIMULATION 
tspan = (0., 15.)
vms_compiled = compile(vms)

q_init = zeros(24)
q_init[21] = 1.2
q = (q_init, zero_q(vms_compiled.virtual_mechanism)) # Robot joint angle, vm joint angles
q̇ = (zero_q̇(vms_compiled.robot), zero_q̇(vms_compiled.virtual_mechanism)) # Robot joint velocity, vm joint velocities

g = VMRobotControl.DEFAULT_GRAVITY
dcache = new_dynamics_cache(vms_compiled)
prob = get_ode_problem(dcache, g, q, q̇, tspan)
sol = solve(prob, Rosenbrock23(autodiff=false), progress=true; maxiters=1e6, abstol=1e-3, reltol=1e-3);

In [83]:
fig = Figure(; size=(720, 720), figure_padding=0)
display(fig)
ls = LScene(fig[1, 1]; show_axis=false)
cam = cam3d!(ls; center=true)
cam.lookat[] = [0.0, 0., 0.22]
cam.eyeposition[] = [0.6, -0.1, 0.42]

plotting_t = Observable(0.0)
plotting_kcache = Observable(new_kinematics_cache(compile(vms)))
robotvisualize!(ls, plotting_kcache)

plotting_vm_kcache = map(plotting_kcache) do k
    VMRobotControl.virtual_mechanism_cache(k)
end
robotsketch!(ls, plotting_vm_kcache; scale = 0.05)

X = SVector(1., 0., 0.)
cylinder = Cylinder(Point3f(cylinder_position + (cylinder_length/2)*X), Point3f(cylinder_position - (cylinder_length/2)*X), cylinder_radius)
mesh!(ls, cylinder; color=:magenta, transparency=true)

savepath = joinpath(module_path, "docs/src/assets/lab_meeting_anim_1.mp4")
display(fig)
animate_robot_odesolution(fig, sol, plotting_kcache, savepath;fps=20,  t=plotting_t);

LoadError: Screen not open!

### Building a Cylinder

In [143]:
c = Mechanism{Float64}("Cylinder")


add_frame!(c; id="prism_frame")
add_joint!(c, Prismatic(SVector(1.0,0.0,0.0)); parent=root_frame(c), child="prism_frame", id="prism_joint")
add_frame!(c; id="revo_frame")
add_joint!(c, Revolute(SVector(1.0,0.0,0.0)); parent="prism_frame", child="revo_frame", id = "revo_joint")
add_frame!(c; id="ee_frame")
add_joint!(c, Rigid(Transform(SVector(0.0,0.0,0.4))); parent ="revo_frame", child ="ee_frame", id = "fixed_joint")

add_coordinate!(c, FrameOrigin("ee_frame"); id="ee position")
add_component!(c, PointMass(0.01, "ee position"); id="ee mass")

add_coordinate!(c, JointSubspace("prism_joint"); id="prism_joint")
add_component!(c, LinearDamper(0.1, "prism_joint"); id="prism_joint_damper") 
add_coordinate!(c, JointSubspace("revo_joint"); id="revo_joint")
add_component!(c, LinearDamper(0.01, "revo_joint"); id="revo_joint_damper") 

add_deadzone_springs!(c, 50.0, (0.0, 1.0), "prism_joint")

add_gravity_compensation!(c, VMRobotControl.DEFAULT_GRAVITY)

vms = VirtualMechanismSystem("cylinderVMS", c)

VirtualMechanismSystem 'cylinderVMS' with robot `Cylinder`, and virtual mechanism `cylinderVMS_virtual_mechanism`.

In [147]:
using Random

function disturbance_func(t)
    Δt = 1.0
    seed_offset = 0
    interval = floor(Int, t / Δt)

    cycle = mod(interval, 4)

    if cycle == 0
        return 0.1 * SVector(0.0, -1.0, 0.0)   # left
    elseif cycle == 2
        return 0.1 * SVector(0.0, 1.0, 0.0)    # right
    else
        # Deterministic random depending on interval + seed
        rng = MersenneTwister(interval + seed_offset)
        v = SVector(randn(rng), randn(rng), randn(rng))
        return 0.1 * normalize(v)
    end
end


f_setup(cache) = get_compiled_coordID(cache, ".robot.ee position")

function f_control(cache, t, args, extra)
    tcp_pos_coord_id = args
    F = disturbance_func(t)
    uᵣ, uᵥ = get_u(cache)
    z = configuration(cache, tcp_pos_coord_id)
    J = jacobian(cache, tcp_pos_coord_id)
    mul!(uᵣ, J', F)
    nothing
end

# SIMULATION
tspan = (0., 12.)
vms_compiled = compile(vms)
q = (zero_q(vms_compiled.robot), zero_q(vms_compiled.virtual_mechanism)) 
q̇ = (zero_q̇(vms_compiled.robot), zero_q̇(vms_compiled.virtual_mechanism)) 
g = VMRobotControl.DEFAULT_GRAVITY
dcache = new_dynamics_cache(vms_compiled)
prob = get_ode_problem(dcache, g, q, q̇, tspan; f_setup, f_control)
sol = solve(prob, Tsit5(), progress=true; maxiters=1e6, abstol=1e-6, reltol=1e-6);

In [150]:
# ANIMATION
fig = Figure(size=(700, 750))
ls = LScene(fig[1, 1]; show_axis = false)  # 3D interactive scene
cam = cam3d!(ls, camera=:perspective, center=false)  
cam.lookat[] = [0.2, 0.2, 0.2]
cam.eyeposition[] = [-1.25, 0.88, 0.7]

plotting_t = Observable(0.0)
plotting_kcache = Observable(new_kinematics_cache(compile(c)))
robotsketch!(ls, plotting_kcache; scale = 0.5)

tcp_pos_id = get_compiled_coordID(plotting_kcache[], "ee position")
tcp_pos = map(plotting_kcache) do kcache
    Point3f(configuration(kcache, tcp_pos_id))
end
force = map(t -> 0.5 * Vec3f(disturbance_func(t)), plotting_t)
arrowsize = map(f -> 0.1*(f'*f)^(0.25), force)
arrows!(ls, map(p -> [p], tcp_pos), map(f -> [f], force); color = :red, arrowsize)

savepath = joinpath(module_path, "docs/src/assets/lab_meeting_anim_2.mp4")
display(fig)
animate_robot_odesolution(fig, sol, plotting_kcache, savepath; fps = 25, t=plotting_t);